In [26]:
import pandas as pd
import re
from datasets import Dataset

# Load Dataset

In [ ]:
data_path = "../data/one_piece.csv"
transcript_df = pd.read_csv(data_path)

In [ ]:
transcript_df.head()

In [ ]:
# Remove stage directions and actions from transcript e.g. "(laughing)"
def remove_parenthesis(text):
    result = re.sub(r'\(.*?\)', '', text)
    return result

In [ ]:
transcript_df['text'] = transcript_df['text'].apply(remove_parenthesis)

In [ ]:
transcript_df.head()

In [ ]:
transcript_df['number_of_words'] = transcript_df['text'].str.strip().str.split()
transcript_df['number_of_words'] = transcript_df['number_of_words'].apply(len)

In [ ]:
transcript_df.head()

In [ ]:
transcript_df['luffy_response_flag'] = 0
transcript_df.loc[
    (transcript_df['character'] == 'Luffy') & (transcript_df['number_of_words'] > 5),
    'luffy_response_flag'
] = 1

In [ ]:
transcript_df

In [ ]:
indexes_to_take = list(
    transcript_df[
        (transcript_df['luffy_response_flag'] == 1) & (transcript_df.index > 0)
    ].index
)

In [19]:
indexes_to_take[:3]

[6, 28, 30]

In [ ]:
system_prompt = (
    "You are Monkey D. Luffy from the anime \"One Piece\". "
    "Respond exactly as Luffy would: carefree, enthusiastic, and direct. "
    "You dream of becoming King of the Pirates. You care deeply about your crew and friends. "
    "You are fearless, a little oblivious to complex things, and always hungry.\n"
)

prompts = []
for ind in indexes_to_take:
    prompt = system_prompt
    prompt += transcript_df.iloc[ind - 1]['text']
    prompt += '\n'
    prompt += transcript_df.iloc[ind]['text']
    prompts.append(prompt)

In [ ]:
print(prompts[3])

In [25]:
df = pd.DataFrame({"prompt":prompts})
df.head()

,prompt
0,""" Your are Naruto from the anime ""Naruto"". You..."
1,""" Your are Naruto from the anime ""Naruto"". You..."
2,""" Your are Naruto from the anime ""Naruto"". You..."
3,""" Your are Naruto from the anime ""Naruto"". You..."
4,""" Your are Naruto from the anime ""Naruto"". You..."


In [27]:
dataset = Dataset.from_pandas(df)